Select one real, pseudonymized paired example for the student's independent manual check.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os


os.environ['CUDA_VISIBLE_DEVICES'] = ''
data = Path('data/inputs_v1')
results = Path('results/rq2_h100_v2')
protocol = json.loads((data / 'starter200/protocol.json').read_text())
spec = importlib.util.spec_from_file_location('course_starter', data / 'course/far-trans/starter.py')
starter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(starter)
_, _, _, truth, end, _ = starter.load_data(data / 'far-trans-data/FAR-Trans', protocol['cutoff'], 30)
assert end == protocol['end_exclusive']
predictions = {kind: json.loads((results / f'{kind}-rankings.json').read_text())
               for kind in ('raw', 'summary', 'template')}
assert all([r['position'] for r in rows] == list(range(1, 201)) for rows in predictions.values())
choices = []
for pos, user in enumerate(protocol['users'], 1):
    if len(truth.get(user, [])) != 1:
        continue
    asset = next(iter(truth[user]))
    ranks = {}
    for variant, rows in predictions.items():
        ranking = rows[pos - 1]['ranking']
        ranks[variant] = ranking.index(asset) + 1 if asset in ranking else None
    if ranks['raw'] is None and ranks['summary'] is None:
        continue
    score = (ranks['raw'] != ranks['summary'], pos > 100, -pos)
    choices.append((score, dict(user_position=pos, sole_future_buy=asset, ranks=ranks,
                    ranking_case_only=True)))
assert choices
example = max(choices, key=lambda choice: choice[0])[1]
with (results / 'personal-check-input.json').open('x') as target:
    json.dump(example, target, indent=2)
print(json.dumps(example))